# 🚀 Prometheus Quant Engine: Quickstart
Welcome to the Prometheus Quant Engine API.

This notebook demonstrates how to price standard European options using our C++ HPC cluster directly from Python. 

### 🔓 Zero Setup Required
To eliminate friction, this notebook is pre-configured with our **Public Demo Key** (`pmt_live_public_colab_demo`). 
This key allows you to execute synchronous Monte Carlo matrices up to **1,000,000 paths** without registering. 

### ⚡ The Computation
We will price a European Call option evaluating 1,000,000 stochastic trajectories. Because European options are path-independent, the temporal resolution is strictly `M = 1`.

In [ ]:
import requests
import uuid
import time

# The Public Demo Key. Limited to 1,000,000 computational steps.
API_KEY = "pmt_live_public_colab_demo" 
BASE_URL = "https://api.prometheusquantengine.com/api/v1/simulations"

# Generate a unique Idempotency Key (UUIDv4) for distributed safety
idem_key = str(uuid.uuid4())

headers = {
    "X-API-Key": API_KEY,
    "Idempotency-Key": idem_key,
    "Content-Type": "application/json"
}

payload = {
    "simulation_type": "European",
    "s_0": 100.0,
    "strike": 100.0,
    "volatility": 0.20,
    "time_to_maturity": 1.0,
    "risk_free_rate": 0.05,
    "option_type": "Call",
    "n_simulations": 1000000, # 1 Million Paths
    "label": "European_Call_Public_Demo"
}

print("Dispatching 1 Million paths to the C++ Core...")
start_time = time.time()
response = requests.post(BASE_URL, json=payload, headers=headers)
print(f"Computed in {time.time() - start_time:.4f} seconds!\n")

data = response.json()
print(f"Fair Value:  {data.get('fair_value')}")
print(f"Delta (Δ):   {data.get('delta')}")
print(f"Ledger Cost: {data.get('credits_cost')} Cr")

### 🛡️ Idempotency: Double-Spend Protection
In a distributed system, network timeouts can cause algorithmic scripts to retry the same payload, resulting in double billing. 

Prometheus enforces strict **Idempotency** via `O(1)` Redis lookups. If we resubmit the exact same payload with the same `Idempotency-Key`, the engine bypasses the C++ workers and returns the cached matrix at **0.00 Cost**.

In [ ]:
print("Re-submitting the exact same request over the network...")
retry_response = requests.post(BASE_URL, json=payload, headers=headers)
retry_data = retry_response.json()

print(f"Fair Value:  {retry_data.get('fair_value')}")
print(f"Ledger Cost: {retry_data.get('credits_cost')} Cr (Cached & Free)")

### 📈 Next Step: Breaking the Speed Limit
The public demo key is hard-capped at 1M paths. Institutional pricing often requires evaluating 50M to 250M paths for path-dependent derivatives (Asian/Barrier options).

**To run Notebooks 02 and 03, you will need a dedicated node:**
1. Head to **[prometheusquantengine.com](https://prometheusquantengine.com)**.
2. Register for API access.
3. You will instantly receive **50 Free Compute Credits** (Sufficient for ~12.5 Billion stochastic steps). No credit card required.